In [24]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
import random
import json
import time
import re

gsm8k = load_dataset("openai/gsm8k", "main", split="test")
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")

In [25]:
random.seed(42)
gsm_idx = random.sample(range(len(gsm8k)), 5)
arc_idx = random.sample(range(len(arc)), 5)
print(len(gsm8k), len(arc))

1319 1172


In [26]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")
model.eval()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [27]:
def generate(user_prompt, max_new_tokens):
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,do_sample=False, temperature=None, top_p=None, top_k=None)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), len(new_tokens)

In [28]:
def gsm_prompt(q, mode):
    if mode == "direct":
        return f"{q}\nReply with only the final number, nothing else. Provide only the answer."
    return f"{q}\nSolve step by step. At the end write 'Answer: <number>'."
    
def arc_prompt(q, options, mode):
    opts = "\n".join(f"{chr(65+i)}. {o}" for i, o in enumerate(options))
    base = f"{q}\n{opts}\n"
    if mode == "direct":
        return base + "Reply with only the letter of the correct option. Provide only the answer."
    return base + "Think step by step. At the end write 'Answer: <letter>'."


def extract_number(text):
    m = re.search(r"Answer:\s*\$?(-?[\d,]*\.?\d+)", text)
    if m:
        s = m.group(1)
    else:
        nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
        s = nums[-1] if nums else None
    return s.replace(",", "").rstrip(".") if s else None
    
def extract_letter(text):
    m = re.search(r"Answer:\s*\(?([A-E])\b", text)
    if m:
        return m.group(1)
    m = re.search(r"\b([A-E])\b", text)
    return m.group(1) if m else None
    
def same_number(a, b):
    try:
        return float(a) == float(b)
    except (TypeError, ValueError):
        return False

def split_steps(text):
    lines = [s.strip() for s in text.split("\n\n") if s.strip()]
    return [s for s in lines if not s.startswith("Answer:")]
    
def gold_steps_count(row):
    if row["dataset"] != "gsm8k":
        return None
    idx = int(row["question_id"].split("_")[1])
    solution = gsm8k[idx]["answer"].split("####")[0]
    return len([s for s in solution.split("\n") if s.strip()])

In [30]:
rows = []
for i in gsm_idx:
    ex = gsm8k[i]
    gold = ex["answer"].split("####")[-1].strip().replace(",", "")
    for mode in ["direct", "cot"]:
        p = gsm_prompt(ex["question"], mode)
        t0 = time.time()
        txt, n = generate(p, 16 if mode == "direct" else 700)
        t = time.time() - t0
        pred = extract_number(txt)
        rows.append(dict(question_id=f"gsm8k_{i}", dataset="gsm8k", model="Qwen/Qwen2.5-0.5B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=same_number(pred, gold), n_tokens=n,gen_time_s=t))

for i in arc_idx:
    ex = arc[i]
    labels = ex["choices"]["label"]
    gold = chr(65 + labels.index(ex["answerKey"]))
    for mode in ["direct", "cot"]:
        p = arc_prompt(ex["question"], ex["choices"]["text"], mode)
        t0 = time.time()
        txt, n = generate(p, 16 if mode == "direct" else 700)
        t = time.time() - t0
        pred = extract_letter(txt)
        rows.append(dict(question_id=f"arc_{i}", dataset="arc", model="Qwen/Qwen2.5-0.5B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=(pred == gold), n_tokens=n,gen_time_s=t))

df = pd.DataFrame(rows)
df["truncated"] = (df["mode"] == "direct") & (df["n_tokens"] >= 16)|(df["mode"] == "cot") & (df["n_tokens"] >= 700)
df["steps"] = df.apply(lambda r: json.dumps(split_steps(r["generated_text"]) if r["mode"] == "cot" else [], ensure_ascii=False), axis=1)
df["n_steps"] = df["steps"].apply(lambda s: len(json.loads(s)))
df["gold_n_steps"] = df.apply(gold_steps_count, axis=1)
df.to_csv("/kaggle/working/results_test10.csv", index=False)
df

,question_id,dataset,model,mode,prompt,generated_text,extracted_answer,gold_answer,correct,n_tokens,gen_time_s,truncated,steps,n_steps,gold_n_steps
0,gsm8k_1309,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,direct,The girls are trying to raise money for a carn...,$1550,1550,2280,False,6,0.368650,False,[],0,3.0
1,gsm8k_1309,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,cot,The girls are trying to raise money for a carn...,To find out how much money each girl raised an...,2780,2280,False,674,31.867860,False,"[""To find out how much money each girl raised ...",19,3.0
2,gsm8k_228,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,direct,Kalinda is working on a 360 piece puzzle with ...,12,12,1,False,3,0.224537,False,[],0,4.0
3,gsm8k_228,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,cot,Kalinda is working on a 360 piece puzzle with ...,To determine how long it will take Kalinda and...,60,1,False,451,21.338355,False,"[""To determine how long it will take Kalinda a...",8,4.0
4,gsm8k_51,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,direct,Tom's ship can travel at 10 miles per hour. H...,2 hours,2,5,False,3,0.223912,False,[],0,3.0
5,gsm8k_51,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,cot,Tom's ship can travel at 10 miles per hour. H...,To determine how long it takes Tom to return f...,26,5,False,605,28.396722,False,"[""To determine how long it takes Tom to return...",14,3.0
6,gsm8k_563,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,direct,James decides to buy birthday candles for his ...,$6,6,12,False,3,0.251592,False,[],0,4.0
7,gsm8k_563,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,cot,James decides to buy birthday candles for his ...,"To determine how much James spends on candles,...",4.8,12,False,290,13.777591,False,"[""To determine how much James spends on candle...",6,4.0
8,gsm8k_501,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,direct,Mariah’s grandma was teaching her to knit. Mar...,98.0,98.0,273,False,5,0.313507,False,[],0,3.0
9,gsm8k_501,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,cot,Mariah’s grandma was teaching her to knit. Mar...,To determine how many yards of yarn Mariah and...,273,273,True,382,17.956280,False,"[""To determine how many yards of yarn Mariah a...",5,3.0


In [31]:
for _, r in df.iterrows():
    print("="*60, r.dataset, r["mode"], "| верно:", r.correct)
    print(r.generated_text)

============================================================ gsm8k direct | верно: False
$1550
============================================================ gsm8k cot | верно: False
To find out how much money each girl raised and then sum them up, we'll follow these steps:

1. **Determine how much money each girl raised:**
   - Let's denote:
     - \( S \) as the amount of money Sarah raised.
     - \( A \) as the amount of money Alexandra raised.
     - \( K \) as the amount of money Kim raised.

   From the problem statement:
   - \( A = 430 \)
   - \( K = 320 + A \)

   Substituting \( A \):
   - \( K = 320 + 430 = 750 \)

   Now, let's calculate the amounts raised by the other two girls:
   - \( M \) is the amount of money Maryam raised.
   - \( R \) is the amount of money Richard raised.

   We know:
   - \( R = 300 \)
   - \( M = R + 400 \)

   Substituting \( R \):
   - \( M = 300 + 400 = 700 \)

2. **Sum up the total amount of money raised:**

   The total amount raised by all t